# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">RAG in 10 Lines of Code</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know the basics of Retrieval-Augmented Generation (RAG), <br>
        ... know how to clean and chunk source documents for RAG with LangChain, <br>
        ... know how to build your own vector database as a retriever with LangChain, <br>
        ... know how to chain a retriever and an LLM into a RAG system with LangChain.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [LangChain AI (2024) Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/rewrite.ipynb?ref=blog.langchain.dev)

## Where We Are

* **5.02:** RAG from scratch.
* **Now:** the same with LangChain, in about 10 lines.
* **5.04 and 5.05:** two ways to make it better.

![RAG extensions](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/rag_extensions.png)

(Source: [Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981))

There are many ways to improve RAG ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)). We start with the simple version.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv

In [ ]:
import os
import re
import urllib.request

from IPython.display import display, Markdown

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# downloads the documents, if they are not here yet
BASE_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + path, path)

### Connect to the AI-Gateway

The same setup as in 4.03: your own computer, connected to the university network or the VPN, and your AI-Gateway key.

We use two models on the gateway:

* a **chat model** that writes the answers (the generator), and
* an **embedding model** that turns texts into vectors (the retriever).

**Good to know:** the gateway allows about 10 requests per minute per model. If a cell pauses, the client is waiting for the next minute and then continues by itself.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")

GENERATOR_NAME = "gwdg.qwen3.8-27b-no-think"  # chat model: writes the answers
RETRIEVER_NAME = "gwdg.qwen3-embedding-4b"    # embedding model: turns texts into vectors

The key comes from the `.env` file in the repo folder (set up in 4.03). If the cell says that no key was found, copy `.env.example` to `.env` and paste your key behind `AI_GATEWAY_KEY=`.

## Loading the Documents

The Markdown files from 5.02, with the links removed.

In [ ]:
markdown_documents_path = "markdown_documents"

In [ ]:
def remove_markdown_links(text: str) -> str:
    # 1. Remove inline links: [text](url)
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # 2. Remove reference-style links: [text][ref]
    text = re.sub(r'\[([^\]]*)\]\[[^\]]*\]', r'\1', text)

    # 3. Remove malformed/broken bracket groups like [[][1][]]
    text = re.sub(r'\[+\s*\]+', '', text)

    # 4. Remove reference definitions like: [1]: https://...
    text = re.sub(r'^\s*\[[^\]]+\]:.*$', '', text, flags=re.MULTILINE)

    return text

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, remove_markdown_links(file.read())])

## RAG in 10 Lines of Code

**In short: LangChain has a building block for every step of 5.02.**

| Step in 5.02 | LangChain |
|---|---|
| chunking | `RecursiveCharacterTextSplitter` |
| embedding model | `OpenAIEmbeddings` |
| vector database | `FAISS` |
| prompt | `PromptTemplate` |
| generator | `ChatOpenAI` |
| putting it all together | a chain with `\|`, as in 5.01 |

In the chain, the retriever fills `{context}`, and `RunnablePassthrough()` passes the question on to `{question}` unchanged. `format_docs` joins the texts of the retrieved chunks into one string.

Embedding all chunks takes about a minute.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
all_splits = text_splitter.create_documents(texts=[x[1] for x in md_files], metadatas=[{"source": x[0]} for x in md_files])
retriever_model = OpenAIEmbeddings(model=RETRIEVER_NAME, base_url=AI_GATEWAY_BASE, api_key=API_KEY, check_embedding_ctx_length=False)
retriever = FAISS.from_documents(all_splits, embedding=retriever_model).as_retriever(search_kwargs={"k": 3})
prompt = PromptTemplate.from_template("You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:")
llm = ChatOpenAI(model=GENERATOR_NAME, base_url=AI_GATEWAY_BASE, api_key=API_KEY, temperature=0)
format_docs = lambda docs: "\n\n".join(doc.page_content for doc in docs)
chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
print("Answer:", chain.invoke("Who plays Daenerys Targaryen?"))

### Save the vector store

Embedding all chunks costs time and about 150,000 tokens of your gateway budget. So we save the vector store in the folder `faiss_index`. The next notebooks load it from there instead of embedding everything again.

In [ ]:
retriever.vectorstore.save_local("faiss_index")

## Your Turn: Take the 10 Lines Apart

Rebuild the chain step by step: copy the matching lines from the 10-line cell into each cell below. Then complete `invoke_basic_rag_chain()`: build the chain with `|` and **return** its answer.

**Tip for the vector store:** do not embed everything again. Load the saved one with `FAISS.load_local("faiss_index", retriever_model, allow_dangerous_deserialization=True)`. (Loading is safe here, because it is our own file.)

### 1. Chunking

In [ ]:
# TODO: split the documents into chunks

### 2. The vector store and the retriever

In [ ]:
# TODO: the embedding model, the vector store, and the retriever

### 3. The generator

In [ ]:
# TODO: the generator

### 4. The prompt template

In [ ]:
# TODO: the prompt template

### 5. The chain

In [ ]:
def invoke_basic_rag_chain(question):
    # TODO: build the chain with | and return its answer
    pass

### Test it

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_basic_rag_chain(question)
print("Answer:", answer)

## What You Learned

* LangChain has a building block for every RAG step: splitter, embeddings, vector store, prompt template, chat model.
* `FAISS` stores the vectors, and `.as_retriever()` turns it into a retriever that returns the k most similar chunks.
* The `|` operator chains retriever, prompt, chat model, and output parser into one RAG chain. `RunnablePassthrough()` passes the question through.
* Save the vector store with `save_local()`, and load it with `load_local()` instead of embedding everything again.

Next: **5.04 RAG Reranking**. Better chunks for the generator.